# Model Building Pipeline — PropNavigator

A step-by-step walkthrough of the **implemented** model pipeline. Each step here calls the
same functions used in production (`src/model_building/*.py`), so this notebook stays in sync
with the code — it's a readable mirror, not a separate copy.

**Input:** `data/pp/preprocessed_properties.csv` (39,065 × 35)
**Target:** `price_in_cr` (trained on `log(price)`, reported back in crores)

**The pipeline:**
1. Train/test split (stratified, log target)
2. Detect numeric vs categorical columns
3. Build preprocessors (encode / scale — fit on train only = leakage-safe)
4. Evaluate 4 base models
5. Hyperparameter tuning (RandomizedSearchCV)
6. Stacking ensemble (top 3)
7. Pick the best + save (MAPE-gated, versioned)
8. Error analysis

> ⏱️ Steps 1–4 are quick. **Step 5 (tuning) is the slow part (~20–40 min).** Run it when you
> want the full result, or just read it to see the logic.

## Setup

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('../..'))   # so `src` is importable from this notebook
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error

# the actual production functions
from src.model_building.mb_main import create_train_test_split
from src.model_building.mb_preprocessing import (
    get_feature_lists, get_tree_preprocessor, get_catboost_preprocessor,
    inverse_transform_target,
)
from src.model_building.mb_evaluation import scorer
from src.model_building.mb_tuning import tune_model
from src.model_building.mb_persistence import save_model
from src.model_building.mb_error_analysis import run_error_analysis

df = pd.read_csv('../../data/pp/preprocessed_properties.csv')
print('data:', df.shape)

## Step 1 — Train/test split (stratified, log target)

**Why stratified:** we chop price into 5 bands (cheap → luxury) and make sure train *and* test
get the same mix — otherwise all the mansions could land in one set and skew the score.
**Why log:** prices span ₹0.3–150 Cr; `log` compresses that so a few mansions don't dominate.
The target `y` is already log-transformed inside `create_train_test_split`.

In [ ]:
X_train, X_test, y_train_log, y_test_log = create_train_test_split(df)
print('train:', X_train.shape, '| test:', X_test.shape)

## Step 2 — Detect numeric vs categorical columns

Derived from dtypes, so nothing hard-codes column names — if features change, this adapts.

In [ ]:
num_features, cat_features = get_feature_lists(X_train)
print(f'numeric ({len(num_features)}):', num_features)
print(f'categorical ({len(cat_features)}):', cat_features)

## Step 3 — Preprocessors (leakage-safe)

Each model type gets the right treatment, and it all lives *inside* the pipeline so it's fit on
**training data only**:
- **Tree models** (RF, XGBoost, LightGBM): ordinal-encode categories, no scaling (trees don't care).
- **CatBoost**: handles categories natively.
- (Linear/SVR would scale + one-hot — but SVR was dropped.)

In [ ]:
tree_preprocessor = get_tree_preprocessor(num_features, cat_features)
catboost_preprocessor = get_catboost_preprocessor(num_features, cat_features)
tree_preprocessor

## Step 4 — Evaluate 4 base models (untuned)

`scorer` builds a pipeline, does 5-fold CV on train, then reports train & test metrics on the
**real price scale** (after inverse-log). This is the baseline to beat.

In [ ]:
model_dict = {
    'RandomForest': (RandomForestRegressor(random_state=42), tree_preprocessor),
    'XGBoost':      (XGBRegressor(random_state=42, objective='reg:squarederror',
                                  tree_method='hist'), tree_preprocessor),
    'LightGBM':     (LGBMRegressor(random_state=42, verbose=-1), tree_preprocessor),
    'CatBoost':     (CatBoostRegressor(random_seed=42, verbose=0,
                                       allow_writing_files=False), catboost_preprocessor),
}

results = []
for name, (model, pre) in model_dict.items():
    results.append(scorer(name, model, pre, X_train, X_test, y_train_log, y_test_log))

results_df = pd.DataFrame(results).sort_values('Test MAPE')
results_df

## Step 5 — Hyperparameter tuning (RandomizedSearchCV)  ⏱️ slow

For each model, try **25 random settings × 3-fold CV**, keep the best. Random search (not grid)
because it finds near-best settings far faster. `tune_model` returns the best fitted pipeline.

> This cell is the long one (~20–40 min for all four). Comment models out to run fewer.

In [ ]:
tuned_models = {}
for name, (model, _) in model_dict.items():
    tuned_models[name] = tune_model(
        model_name=name, model=model,
        X_train=X_train, y_train_log=y_train_log,
        X_test=X_test, y_test_log=y_test_log,
        numerical_features=num_features, categorical_features=cat_features,
    )
print('tuned:', [k for k, v in tuned_models.items() if v is not None])

In [ ]:
# tuned test MAPE (real price scale) for each model
y_true = inverse_transform_target(y_test_log)
candidate = {}
for name, pipe in tuned_models.items():
    if pipe is None:
        continue
    mape = mean_absolute_percentage_error(y_true, inverse_transform_target(pipe.predict(X_test))) * 100
    candidate[name] = (pipe, mape)
    print(f'{name:14s} tuned Test MAPE: {mape:.2f}%')

## Step 6 — Stacking ensemble (top 3)

Take the best 3 tuned models and add a **meta-model** (Ridge) that learns how to blend their
predictions — "three experts + a fourth who knows when to trust each." CatBoost is excluded
from stacking (its params don't play nicely with sklearn's `clone()` that stacking needs).

In [ ]:
CLONE_INCOMPATIBLE = {'CatBoost'}
stackable = {n: (p, m) for n, (p, m) in candidate.items() if n not in CLONE_INCOMPATIBLE}

ranked = sorted(stackable.items(), key=lambda x: x[1][1])          # by MAPE
top_estimators = [(n, p) for n, (p, _) in ranked[:3]]
print('stacking:', [n for n, _ in top_estimators])

stacking = StackingRegressor(estimators=top_estimators,
                             final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1)
stacking.fit(X_train, y_train_log)
stack_mape = mean_absolute_percentage_error(
    y_true, inverse_transform_target(stacking.predict(X_test))) * 100
print(f'Stacking Test MAPE: {stack_mape:.2f}%')

## Step 7 — Pick the best & save (MAPE-gated, versioned)

Choose the lowest test-MAPE model. `save_model` only overwrites the saved model **if the new one
is actually better**, keeps a timestamped version, and logs every run to
`artifacts/experiment_log.csv`. We also store residual quantiles → the app's price *range*.

In [ ]:
candidate['Stacking'] = (stacking, stack_mape)
best_name, (best_pipe, best_mape) = min(candidate.items(), key=lambda x: x[1][1])
print(f'Best model: {best_name} ({best_mape:.2f}% MAPE)')

# residual quantiles for the prediction interval
y_pred = inverse_transform_target(best_pipe.predict(X_test))
pct = (y_true - y_pred) / y_pred
rq = {'q05': float(np.percentile(pct, 5)),  'q95': float(np.percentile(pct, 95)),
      'q10': float(np.percentile(pct, 10)), 'q90': float(np.percentile(pct, 90))}
print('90% interval:', round(rq['q05'], 3), 'to', round(rq['q95'], 3))

save_model(model_pipeline=best_pipe, model_name=best_name, metric=round(best_mape, 2),
           filepath='../../artifacts/best_model.joblib', residual_quantiles=rq)

## Step 8 — Error analysis

Break the error down by **price bracket, property type, and sector** — so we know *where* the
model is reliable, not just its average. (Fixed: the top price bracket is now open-ended so
luxury homes are included.)

In [ ]:
res = run_error_analysis(best_pipe, X_test, y_test_log, output_dir='../../data/error_analysis')

print('=== SUMMARY ==='); print(res['summary'].to_string(index=False))
seg = res['segment_df']
print('\n=== BY PRICE BRACKET ===')
print(seg[seg['segment_type'] == 'price_bracket'].to_string(index=False))
print('\n=== BY PROPERTY TYPE ===')
print(seg[seg['segment_type'] == 'property_type'].to_string(index=False))

---
### Shortcut: run the whole thing at once
Everything above is exactly what `run_model_building` does end-to-end:
```python
from src.model_building.mb_main import run_model_building
run_model_building(df)
```